# ICU Mortality Prediction & Algorithmic Fairness
**Master's Thesis 

### Sections
1. Setup & data loading
2. Cohort construction
3. Demographic analysis (EDA)
4. Missingness analysis by demographic group
5. Feature engineering
6. 5-fold cross-validation. all three models (transformer comes later)
7. Fairness evaluation
8. Investigation, why do disparities exist?
9. Bootstrap confidence intervals
10. Summary of findings

---
## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
import os
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.3f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

DATA_DIR = Path('/home/ino/Data/physionet.org/files/mimiciv/3.1')
ICU_DIR  = DATA_DIR / 'icu'
HOSP_DIR = DATA_DIR / 'hosp'
OUT_DIR  = Path('/home/ino/thesis_outputs')
OUT_DIR.mkdir(exist_ok=True)

VITALS = {
    'heart_rate':  220045,
    'resp_rate':   220210,
    'spo2':        220277,
    'temperature': 223762,
    'map':         220052,
}

VITAL_BOUNDS = {
    'heart_rate':  (20, 300),
    'resp_rate':   (4,  60),
    'spo2':        (50, 100),
    'temperature': (25, 45),
    'map':         (20, 200),
}

N_FOLDS      = 5
RANDOM_STATE = 42
print('Setup complete.')
print(f'Output directory: {OUT_DIR}')

In [ ]:
print('Loading tables...')
icu_stays  = pd.read_csv(ICU_DIR  / 'icustays.csv.gz')
admissions = pd.read_csv(HOSP_DIR / 'admissions.csv.gz')
patients   = pd.read_csv(HOSP_DIR / 'patients.csv.gz')

icu_stays['intime']     = pd.to_datetime(icu_stays['intime'])
icu_stays['outtime']    = pd.to_datetime(icu_stays['outtime'])
admissions['deathtime'] = pd.to_datetime(admissions['deathtime'])

print(f'ICU stays:   {len(icu_stays):,}')
print(f'Admissions:  {len(admissions):,}')
print(f'Patients:    {len(patients):,}')

---
## 2. Cohort Construction

- First ICU stay per patient only (prevents data leakage from re-admissions)
- Adults aged 18 or older
- Minimum 4-hour ICU stay
- Race/ethnicity known (Unknown excluded from fairness analysis)

In [ ]:
cohort = icu_stays.sort_values('intime').groupby('subject_id').first().reset_index()
print(f'After first-stay filter:     {len(cohort):,}')

cohort = cohort.merge(patients[['subject_id', 'anchor_age', 'gender']], on='subject_id')
cohort = cohort[cohort['anchor_age'] >= 18]
print(f'After adult filter:          {len(cohort):,}')

cohort['los_hours'] = (cohort['outtime'] - cohort['intime']).dt.total_seconds() / 3600
cohort = cohort[cohort['los_hours'] >= 4]
print(f'After 4h stay filter:        {len(cohort):,}')

cohort = cohort.merge(admissions[['hadm_id', 'hospital_expire_flag', 'race']], on='hadm_id')
print(f'After admissions merge:      {len(cohort):,}')

In [ ]:
def consolidate_race(race_str):
    if pd.isna(race_str): return 'Unknown'
    r = race_str.upper()
    if 'WHITE' in r:   return 'White'
    elif 'BLACK' in r or 'AFRICAN' in r: return 'Black'
    elif 'HISPANIC' in r or 'LATINO' in r: return 'Hispanic'
    elif 'ASIAN' in r: return 'Asian'
    elif 'UNKNOWN' in r or 'UNABLE' in r or 'DECLINED' in r: return 'Unknown'
    else: return 'Other'

cohort['race_group'] = cohort['race'].apply(consolidate_race)
cohort = cohort[cohort['race_group'] != 'Unknown']  # exclude unknown race

cohort['age_group'] = pd.cut(
    cohort['anchor_age'], bins=[17, 40, 60, 75, 120],
    labels=['18-40', '41-60', '61-75', '75+']
)
cohort['gender'] = cohort['gender'].map({'M': 'Male', 'F': 'Female'})

print(f'Final cohort size: {len(cohort):,}')
print(f'Mortality rate:    {cohort["hospital_expire_flag"].mean():.1%}')
print(cohort['race_group'].value_counts().to_string())
cohort.to_csv(OUT_DIR / 'cohort.csv', index=False)
print('Saved cohort.csv')

---
## 3. Demographic Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

race_counts = cohort['race_group'].value_counts()
axes[0].bar(race_counts.index, race_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Cohort by race/ethnicity'); axes[0].set_ylabel('Patients')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(race_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

gender_counts = cohort['gender'].value_counts()
axes[1].bar(gender_counts.index, gender_counts.values, color='seagreen', edgecolor='white')
axes[1].set_title('Cohort by gender'); axes[1].set_ylabel('Patients')
for i, v in enumerate(gender_counts.values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

age_counts = cohort['age_group'].value_counts().sort_index()
axes[2].bar(age_counts.index.astype(str), age_counts.values, color='darkorange', edgecolor='white')
axes[2].set_title('Cohort by age group'); axes[2].set_ylabel('Patients')
for i, v in enumerate(age_counts.values):
    axes[2].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

plt.suptitle('Demographic composition of study cohort', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig1_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig1_demographics.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for col, title, color, ax in [
    ('race_group', 'Race/ethnicity', 'steelblue', axes[0]),
    ('gender',     'Gender',         'seagreen',  axes[1]),
    ('age_group',  'Age group',      'darkorange',axes[2]),
]:
    mort = cohort.groupby(col)['hospital_expire_flag'].agg(['mean','count']).reset_index()
    mort = mort.sort_values('mean', ascending=False)
    bars = ax.bar(mort[col].astype(str), mort['mean']*100, color=color, edgecolor='white')
    overall = cohort['hospital_expire_flag'].mean()*100
    ax.axhline(overall, color='crimson', linestyle='--', linewidth=1.2, label=f'Overall: {overall:.1f}%')
    ax.set_title(f'Mortality rate by {title.lower()}'); ax.set_ylabel('Mortality rate (%)')
    ax.tick_params(axis='x', rotation=30); ax.legend(fontsize=9)
    for bar, (_, row) in zip(bars, mort.iterrows()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'n={row["count"]:,}', ha='center', fontsize=8, color='gray')

plt.suptitle('In-hospital mortality rate by demographic group', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig2_mortality_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig2_mortality_by_group.png')

In [ ]:
summary = cohort.groupby('race_group').agg(
    n=('subject_id','count'),
    mortality_rate=('hospital_expire_flag','mean'),
    mean_age=('anchor_age','mean'),
    pct_female=('gender', lambda x: (x=='Female').mean()),
    mean_los_hours=('los_hours','mean'),
).round(3)
summary['pct_of_cohort'] = (summary['n']/len(cohort)*100).round(1)
summary = summary.sort_values('n', ascending=False)
print(summary.to_string())
summary.to_csv(OUT_DIR / 'table1_cohort_summary.csv')
print('Saved table1_cohort_summary.csv')

---
## 4. Missingness Analysis by Demographic Group

Key hypothesis: if certain groups have fewer vital sign measurements,
the model has less information so a structural bias independent of model architecture.

In [ ]:
print('Reading chartevents. May take several minutes...')

stay_times      = cohort.set_index('stay_id')['intime'].to_dict()
cohort_stay_ids = set(stay_times.keys())
target_ids      = set(VITALS.values())
all_events_list = []

reader = pd.read_csv(
    ICU_DIR / 'chartevents.csv.gz',
    usecols=['stay_id', 'charttime', 'itemid', 'valuenum'],
    chunksize=500_000,
)

for i, chunk in enumerate(reader):
    sub = chunk[
        chunk['itemid'].isin(target_ids) &
        chunk['stay_id'].isin(cohort_stay_ids)
    ].copy()
    if len(sub):
        sub['charttime'] = pd.to_datetime(sub['charttime'])
        sub['intime']    = sub['stay_id'].map(stay_times)
        sub['hours_in']  = (sub['charttime'] - sub['intime']).dt.total_seconds() / 3600
        sub = sub[(sub['hours_in'] >= 0) & (sub['hours_in'] <= 24)].dropna(subset=['valuenum'])
        id_to_name = {v: k for k, v in VITALS.items()}
        sub['vital'] = sub['itemid'].map(id_to_name)
        filtered = []
        for vital, (lo, hi) in VITAL_BOUNDS.items():
            mask = (sub['vital']==vital)&(sub['valuenum']>=lo)&(sub['valuenum']<=hi)
            filtered.append(sub[mask])
        sub = pd.concat(filtered)
        all_events_list.append(sub[['stay_id','vital','valuenum','hours_in']])
    if i % 10 == 0: print(f'  chunk {i}...')

all_events = pd.concat(all_events_list, ignore_index=True)
print(f'\nTotal observations (first 24h, clean): {len(all_events):,}')
all_events.to_parquet(OUT_DIR / 'events_24h.parquet', index=False)
print('Saved events_24h.parquet')

In [ ]:
# Load from parquet if re-running: all_events = pd.read_parquet(OUT_DIR / 'events_24h.parquet')

meas = all_events.groupby(['stay_id','vital']).size().reset_index(name='n_measurements')
meas = meas.merge(cohort[['stay_id','race_group','gender','age_group']], on='stay_id')

avg_race = meas.groupby(['race_group','vital'])['n_measurements'].mean().unstack('vital')
print('Average measurements per patient in first 24h — by race:')
print(avg_race.round(1).to_string())

has_meas = all_events.groupby(['stay_id','vital']).size().reset_index(name='n')
has_meas = has_meas.merge(cohort[['stay_id','race_group']], on='stay_id')
coverage = has_meas.groupby(['race_group','vital'])['n'].count().unstack('vital')
group_n  = cohort['race_group'].value_counts()
cov_pct  = coverage.div(group_n, axis=0)*100
print('\n% with at least one measurement — by race:')
print(cov_pct.round(1).to_string())
cov_pct.to_csv(OUT_DIR / 'table2_vital_coverage_by_race.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vital_colors = ['steelblue','seagreen','darkorange','mediumpurple','crimson']

for col, title, ax in [
    ('race_group','Race/ethnicity',axes[0]),
    ('gender',    'Gender',        axes[1]),
    ('age_group', 'Age group',     axes[2]),
]:
    avg = meas.groupby([col,'vital'])['n_measurements'].mean().unstack('vital')
    avg.plot(kind='bar', ax=ax, color=vital_colors, edgecolor='white', linewidth=0.4)
    ax.set_title(f'Avg measurements by {title.lower()}')
    ax.set_ylabel('Avg measurements per patient')
    ax.set_xlabel(''); ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8, title='Vital')

plt.suptitle('Vital sign measurement frequency by demographic group (first 24h)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig3_measurement_frequency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig3_measurement_frequency.png')

---
## 5. Feature Engineering

Two representations of each patient's first 24h:
- **Tabular (21 features)**: mean/std/min/max × 5 vitals + age for XGBoost
- **Time series (24 × 10)**: hourly bins + missingness mask for LSTM and DNN

In [ ]:
# 5.1 Tabular features
agg = all_events.groupby(['stay_id','vital'])['valuenum'].agg(
    ['mean','std','min','max']
).unstack('vital')
agg.columns = [f'{vital}_{stat}' for stat, vital in agg.columns]
agg = agg.reset_index()
print(f'Tabular feature matrix: {agg.shape}')

In [ ]:
# 5.2 Time series matrix
print('Building time series matrices...')

vital_names   = list(VITALS.keys())
stay_ids_list = cohort['stay_id'].tolist()

all_events['hour_bin'] = np.ceil(all_events['hours_in']).clip(1, 24).astype(int)
hourly      = all_events.groupby(['stay_id','vital','hour_bin'])['valuenum'].mean().reset_index()
hourly_wide = hourly.pivot_table(index='stay_id', columns=['vital','hour_bin'], values='valuenum')
pop_medians = all_events.groupby('vital')['valuenum'].median().to_dict()

n_patients = len(stay_ids_list)
X_ts = np.zeros((n_patients, 24, len(vital_names)), dtype=np.float32)
M_ts = np.zeros((n_patients, 24, len(vital_names)), dtype=np.float32)

for i, sid in enumerate(stay_ids_list):
    if sid not in hourly_wide.index:
        for j, v in enumerate(vital_names):
            X_ts[i, :, j] = pop_medians.get(v, 0)
        continue
    row = hourly_wide.loc[sid]
    for j, vital in enumerate(vital_names):
        series = np.full(24, np.nan)
        for h in range(1, 25):
            key = (vital, h)
            if key in row.index and not pd.isna(row[key]):
                series[h-1] = row[key]
                M_ts[i, h-1, j] = 1.0
        mask_obs = ~np.isnan(series)
        if mask_obs.any():
            last = series[mask_obs][0]
            for h in range(24):
                if not np.isnan(series[h]): last = series[h]
                else: series[h] = last
        series = np.where(np.isnan(series), pop_medians.get(vital, 0), series)
        X_ts[i, :, j] = series
    if i % 5000 == 0: print(f'  {i}/{n_patients} patients...')

print(f'Time series shape: {X_ts.shape}')
np.save(OUT_DIR / 'X_ts.npy', X_ts)
np.save(OUT_DIR / 'M_ts.npy', M_ts)
print('Saved X_ts.npy and M_ts.npy')

In [ ]:
# 5.3 Assemble dataset as numpy arrays
from sklearn.model_selection import StratifiedKFold

df = cohort[['stay_id','hospital_expire_flag','anchor_age',
             'gender','race_group','age_group']].merge(agg, on='stay_id', how='inner')

ts_stay_ids  = np.array(stay_ids_list)
keep_mask    = np.isin(ts_stay_ids, df.stay_id.values)
X_ts_aligned = X_ts[keep_mask]
M_ts_aligned = M_ts[keep_mask]
ts_stay_ids  = ts_stay_ids[keep_mask]
df           = df.set_index('stay_id').loc[ts_stay_ids].reset_index()

feature_cols = [c for c in df.columns
                if c not in ['stay_id','hospital_expire_flag','gender','race_group','age_group']]

# Convert immediately to numpy
X_tab = np.array(df[feature_cols].values, dtype=np.float64)
y     = np.array(df['hospital_expire_flag'].values, dtype=np.int32)
demo  = df[['race_group','gender','age_group']].reset_index(drop=True)

# Stratification label: race + outcome
strat_label = df['race_group'].astype(str) + '_' + df['hospital_expire_flag'].astype(str)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f'Dataset size: {len(y):,} patients')
print(f'Mortality rate: {y.mean():.1%}')
print(f'Features: {len(feature_cols)}')
print(f'Time series shape: {X_ts_aligned.shape}')
print(f'Will run {N_FOLDS}-fold CV stratified on outcome × race group')

---
## 6. Five-Fold Cross-Validation for all three models

Each fold:
1. Splits data into 80% train / 20% test (stratified on outcome × race)
2. Trains XGBoost, Deep Bidirectional LSTM, and Deep Feedforward NN
3. Evaluates overall AUROC/AUPRC and per subgroup fairness metrics

All reported metrics are averages across the 5 folds.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
import joblib

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count()
print(f'Device: {device}  |  GPUs: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# Model definitions

class MortalityLSTM(nn.Module):
    """3-layer Bidirectional LSTM: (batch,24,10) → (batch,)"""
    def __init__(self, input_size=10, hidden_size=128, num_layers=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True,
                            dropout=dropout, bidirectional=True)
        self.layer_norm = nn.LayerNorm(hidden_size * 2)
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size*2, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64),            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        last = self.layer_norm(out[:, -1, :])
        return self.classifier(self.dropout(last)).squeeze(1)


class MortalityDNN(nn.Module):
    """Deep Feedforward NN: (batch,24,10) flattened to (batch,240) → (batch,)"""
    def __init__(self, input_size=240, dropout=0.3):
        super().__init__()
        self.block1 = nn.Sequential(nn.Linear(input_size,512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout))
        self.block2 = nn.Sequential(nn.Linear(512,256),        nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout))
        self.block3 = nn.Sequential(nn.Linear(256,128),        nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(dropout))
        self.block4 = nn.Sequential(nn.Linear(128,64),         nn.BatchNorm1d(64),  nn.ReLU(), nn.Dropout(dropout))
        self.residual_proj = nn.Linear(256, 64)
        self.out = nn.Linear(64, 1)
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.block1(x); x2 = self.block2(x)
        x = self.block4(self.block3(x2)) + self.residual_proj(x2)
        return self.out(x).squeeze(1)


def train_neural_net(model, train_loader, test_loader, y_test_fold,
                     n_epochs, lr, use_cosine=False, label='Model'):
    """Train a neural network and return best probabilities."""
    scale_pw = torch.tensor(
        [(y_test_fold==0).sum()/(y_test_fold==1).sum()],
        dtype=torch.float32
    ).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=scale_pw)

    if use_cosine:
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-5)
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3, factor=0.5)

    best_auroc = 0
    best_probs = None

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        model.eval()
        probs_list = []
        with torch.no_grad():
            for xb, _ in test_loader:
                probs_list.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
        probs = np.concatenate(probs_list)
        auroc = roc_auc_score(y_test_fold, probs)

        if use_cosine:
            scheduler.step()
        else:
            scheduler.step(auroc)

        if auroc > best_auroc:
            best_auroc = auroc
            best_probs = probs.copy()

        if epoch % 10 == 0 or epoch == 1:
            print(f'    {label} ep {epoch:02d}  AUROC={auroc:.4f}{"  *" if auroc==best_auroc else ""}')

    return best_probs, best_auroc


def get_fairness_metrics(probs, y_true, demo_df, group_col, threshold=0.5):
    """Compute fairness metrics for a single group dimension."""
    results = {}
    for g in sorted(demo_df[group_col].dropna().unique()):
        mask = (demo_df[group_col] == g).values
        n    = mask.sum()
        if n < 20 or y_true[mask].sum() < 5: continue
        y_g, p_g = y_true[mask], probs[mask]
        pred  = (p_g >= threshold).astype(int)
        tp = ((pred==1)&(y_g==1)).sum(); fn = ((pred==0)&(y_g==1)).sum()
        fp = ((pred==1)&(y_g==0)).sum(); tn = ((pred==0)&(y_g==0)).sum()
        frac_pos, mean_pred = calibration_curve(y_g, p_g, n_bins=5)
        results[g] = {
            'n':    n,
            'auroc': roc_auc_score(y_g, p_g),
            'auprc': average_precision_score(y_g, p_g),
            'tpr':   tp/(tp+fn) if (tp+fn)>0 else 0,
            'fpr':   fp/(fp+tn) if (fp+tn)>0 else 0,
            'ece':   np.mean(np.abs(frac_pos - mean_pred)),
            'mortality_rate': y_g.mean(),
        }
    return results

print('Model classes and utilities defined.')

In [ ]:
# MAIN CV LOOP
# Stores per-fold results for all models and all subgroups

GROUP_COLS  = ['race_group', 'gender', 'age_group']
MODEL_NAMES = ['XGBoost', 'Deep NN', 'LSTM']

# fold_results[fold][model] = {'overall': {...}, 'race_group': {...}, ...}
fold_results   = []
# Also collect probabilities on the whole dataset via out-of-fold predictions
oof_probs = {m: np.zeros(len(y)) for m in MODEL_NAMES}

print(f'Starting {N_FOLDS}-fold cross-validation...')
print(f'Models: {MODEL_NAMES}')
print(f'Total estimated training time: 8-12 hours on 2x RTX 3090')
print('=' * 60)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_tab, strat_label)):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──────────────────────────────────────')

    # ── Split data ──
    X_tr_tab, X_te_tab = X_tab[train_idx], X_tab[test_idx]
    X_tr_ts,  X_te_ts  = X_ts_aligned[train_idx], X_ts_aligned[test_idx]
    M_tr_ts,  M_te_ts  = M_ts_aligned[train_idx], M_ts_aligned[test_idx]
    y_tr, y_te         = y[train_idx], y[test_idx]
    demo_te             = demo.iloc[test_idx].reset_index(drop=True)

    scale_pw = (y_tr == 0).sum() / (y_tr == 1).sum()

    # ── Impute + scale tabular features ──
    imp = SimpleImputer(strategy='median')
    scl = StandardScaler()
    X_tr_tab_s = scl.fit_transform(imp.fit_transform(X_tr_tab))
    X_te_tab_s = scl.transform(imp.transform(X_te_tab))

    # ── Normalize time series ──
    ts_mean = X_tr_ts.mean(axis=(0,1), keepdims=True)
    ts_std  = X_tr_ts.std(axis=(0,1),  keepdims=True) + 1e-8
    X_tr_ts_n = (X_tr_ts - ts_mean) / ts_std
    X_te_ts_n = (X_te_ts - ts_mean) / ts_std

    # ── Build LSTM/DNN tensors ──
    X_tr_lstm = np.concatenate([X_tr_ts_n, M_tr_ts], axis=2).astype(np.float32)
    X_te_lstm = np.concatenate([X_te_ts_n, M_te_ts], axis=2).astype(np.float32)

    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    y_te_t = torch.tensor(y_te, dtype=torch.float32)

    lstm_train_loader = DataLoader(
        TensorDataset(torch.tensor(X_tr_lstm), y_tr_t),
        batch_size=1024, shuffle=True, num_workers=4, pin_memory=True)
    lstm_test_loader = DataLoader(
        TensorDataset(torch.tensor(X_te_lstm), y_te_t),
        batch_size=1024, shuffle=False, num_workers=4, pin_memory=True)

    fold_probs = {}

    # ────────────────────────────────────────────────────────────────────────
    # MODEL 1: XGBoost
    # ────────────────────────────────────────────────────────────────────────
    print(f'  [XGBoost] training...')
    xgb_fold = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pw,
        eval_metric='logloss', random_state=RANDOM_STATE,
        verbosity=0, device='cuda',
    )
    xgb_fold.fit(X_tr_tab_s, y_tr)
    xgb_p = xgb_fold.predict_proba(X_te_tab_s)[:, 1]
    fold_probs['XGBoost'] = xgb_p
    oof_probs['XGBoost'][test_idx] = xgb_p
    print(f'  [XGBoost] AUROC={roc_auc_score(y_te, xgb_p):.4f}')
    joblib.dump(xgb_fold, OUT_DIR / f'xgb_fold{fold+1}.pkl')

    # ────────────────────────────────────────────────────────────────────────
    # MODEL 2: Deep Bidirectional LSTM
    # ────────────────────────────────────────────────────────────────────────
    print(f'  [LSTM] training for 50 epochs...')
    lstm_base = MortalityLSTM(input_size=10, hidden_size=128, num_layers=3, dropout=0.3)
    lstm_model = nn.DataParallel(lstm_base) if n_gpus > 1 else lstm_base
    lstm_model = lstm_model.to(device)

    lstm_p, lstm_auroc = train_neural_net(
        lstm_model, lstm_train_loader, lstm_test_loader, y_te,
        n_epochs=50, lr=5e-4, use_cosine=False, label='LSTM'
    )
    fold_probs['LSTM'] = lstm_p
    oof_probs['LSTM'][test_idx] = lstm_p
    print(f'  [LSTM] Best AUROC={lstm_auroc:.4f}')
    torch.save(lstm_base.state_dict(), OUT_DIR / f'lstm_fold{fold+1}.pt')
    del lstm_model, lstm_base
    torch.cuda.empty_cache()

    # ────────────────────────────────────────────────────────────────────────
    # MODEL 3: Deep Feedforward NN
    # ────────────────────────────────────────────────────────────────────────
    print(f'  [DNN] training for 80 epochs...')
    dnn_base  = MortalityDNN(input_size=240, dropout=0.3)
    dnn_model = nn.DataParallel(dnn_base) if n_gpus > 1 else dnn_base
    dnn_model = dnn_model.to(device)

    dnn_p, dnn_auroc = train_neural_net(
        dnn_model, lstm_train_loader, lstm_test_loader, y_te,
        n_epochs=80, lr=3e-4, use_cosine=True, label='DNN'
    )
    fold_probs['Deep NN'] = dnn_p
    oof_probs['Deep NN'][test_idx] = dnn_p
    print(f'  [DNN] Best AUROC={dnn_auroc:.4f}')
    torch.save(dnn_base.state_dict(), OUT_DIR / f'dnn_fold{fold+1}.pt')
    del dnn_model, dnn_base
    torch.cuda.empty_cache()

    # ────────────────────────────────────────────────────────────────────────
    # Fairness metrics for this fold
    # ────────────────────────────────────────────────────────────────────────
    fold_record = {'fold': fold+1, 'models': {}}
    for model_name in MODEL_NAMES:
        probs = fold_probs[model_name]
        fold_record['models'][model_name] = {
            'overall': {
                'n':     len(y_te),
                'auroc': roc_auc_score(y_te, probs),
                'auprc': average_precision_score(y_te, probs),
            }
        }
        for col in GROUP_COLS:
            fold_record['models'][model_name][col] = get_fairness_metrics(probs, y_te, demo_te, col)

    fold_results.append(fold_record)

    # Print fold summary
    print(f'\n  Fold {fold+1} summary:')
    print(f'  {"Model":<12} {"AUROC":>8} {"AUPRC":>8}')
    for m in MODEL_NAMES:
        ov = fold_record['models'][m]['overall']
        print(f'  {m:<12} {ov["auroc"]:>8.4f} {ov["auprc"]:>8.4f}')

# Save out-of-fold probabilities
for m in MODEL_NAMES:
    np.save(OUT_DIR / f'oof_probs_{m.replace(" ","_")}.npy', oof_probs[m])

print('\n' + '='*60)
print('Cross-validation complete!')
print('Saved fold model weights and out-of-fold probabilities.')

---
## 7. Fairness Evaluation

We aggregate CV results across all 5 folds to produce stable mean ± std estimates
for every metric, model, and demographic group.

In [ ]:
# Aggregate CV results
records = []
for fr in fold_results:
    fold = fr['fold']
    for model_name in MODEL_NAMES:
        # Overall
        ov = fr['models'][model_name]['overall']
        records.append({'fold':fold,'model':model_name,'dimension':'overall',
                        'group':'Overall', **ov})
        # Per group
        for col in GROUP_COLS:
            for group, metrics in fr['models'][model_name][col].items():
                records.append({'fold':fold,'model':model_name,'dimension':col,
                                'group':group, **metrics})

results_df = pd.DataFrame(records)
results_df.to_csv(OUT_DIR / 'cv_all_results.csv', index=False)

# Summary: mean ± std per model × group
summary = results_df.groupby(['model','dimension','group']).agg(
    mean_auroc=('auroc','mean'),
    std_auroc=('auroc','std'),
    mean_auprc=('auprc','mean'),
    std_auprc=('auprc','std'),
    mean_tpr=('tpr','mean'),
    std_tpr=('tpr','std'),
    mean_n=('n','mean'),
).round(4).reset_index()

summary.to_csv(OUT_DIR / 'cv_summary_all.csv', index=False)
print('Saved cv_all_results.csv and cv_summary_all.csv')

# Print overall performance
print('\n── Overall performance (mean ± std across 5 folds) ──')
print(f'{"Model":<12} {"AUROC":>16} {"AUPRC":>16}')
print('-'*48)
for m in MODEL_NAMES:
    row = summary[(summary['model']==m)&(summary['group']=='Overall')].iloc[0]
    print(f'{m:<12}  {row["mean_auroc"]:.4f} ± {row["std_auroc"]:.4f}  '
          f'{row["mean_auprc"]:.4f} ± {row["std_auprc"]:.4f}')

In [ ]:
# AUROC by race group
print('\n── AUROC by race group (mean ± std) ──')
race_summary = summary[summary['dimension']=='race_group'].sort_values(['model','mean_auroc'])
for m in MODEL_NAMES:
    print(f'\n{m}:')
    sub = race_summary[race_summary['model']==m]
    for _, row in sub.iterrows():
        print(f'  {row["group"]:<12}  {row["mean_auroc"]:.4f} ± {row["std_auroc"]:.4f}  '
              f'TPR={row["mean_tpr"]:.3f} ± {row["std_tpr"]:.3f}  n={row["mean_n"]:.0f}')

In [ ]:
# Fig 6: AUROC by group
colors = {'XGBoost': 'steelblue', 'Deep NN': 'seagreen', 'LSTM': 'darkorange'}
titles = {'race_group': 'Race/ethnicity', 'gender': 'Gender', 'age_group': 'Age group'}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, GROUP_COLS):
    xgb_groups = sorted(summary[(summary['model']=='XGBoost')&(summary['dimension']==col)]['group'].unique())
    x = np.arange(len(xgb_groups))
    width = 0.25

    for offset, m in enumerate(MODEL_NAMES):
        sub = summary[(summary['model']==m)&(summary['dimension']==col)].set_index('group')
        means  = [sub.loc[g,'mean_auroc'] if g in sub.index else np.nan for g in xgb_groups]
        stds   = [sub.loc[g,'std_auroc']  if g in sub.index else 0      for g in xgb_groups]
        bars = ax.bar(x + offset*width, means, width, label=m,
                      color=colors[m], edgecolor='white', alpha=0.85)
        ax.errorbar(x + offset*width, means, yerr=stds,
                    fmt='none', color='black', capsize=3, linewidth=1)

    # Overall lines
    for m in MODEL_NAMES:
        ov = summary[(summary['model']==m)&(summary['group']=='Overall')].iloc[0]
        ax.axhline(ov['mean_auroc'], color=colors[m], linestyle='--', linewidth=0.8, alpha=0.5)

    ax.set_xticks(x + width)
    ax.set_xticklabels(xgb_groups, rotation=30, ha='right')
    ax.set_ylabel('AUROC (mean ± std across 5 folds)')
    ax.set_ylim(0.5, 1.0)
    ax.set_title(f'AUROC by {titles[col].lower()}')
    ax.legend(fontsize=8)

plt.suptitle('Model performance by demographic group — 5-fold CV mean ± std', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig6_auroc_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig6_auroc_by_group.png')

In [ ]:
# Fig 7: TPR by group
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, col in zip(axes, GROUP_COLS):
    xgb_groups = sorted(summary[(summary['model']=='XGBoost')&(summary['dimension']==col)]['group'].unique())
    x = np.arange(len(xgb_groups))
    width = 0.25

    for offset, m in enumerate(MODEL_NAMES):
        sub = summary[(summary['model']==m)&(summary['dimension']==col)].set_index('group')
        means = [sub.loc[g,'mean_tpr'] if g in sub.index else np.nan for g in xgb_groups]
        stds  = [sub.loc[g,'std_tpr']  if g in sub.index else 0      for g in xgb_groups]
        ax.bar(x + offset*width, means, width, label=m, color=colors[m], edgecolor='white', alpha=0.85)
        ax.errorbar(x + offset*width, means, yerr=stds,
                    fmt='none', color='black', capsize=3, linewidth=1)

    ax.set_xticks(x + width)
    ax.set_xticklabels(xgb_groups, rotation=30, ha='right')
    ax.set_ylabel('True positive rate (mean ± std)')
    ax.set_ylim(0, 1)
    ax.set_title(f'Sensitivity by {titles[col].lower()}')
    ax.legend(fontsize=8)

plt.suptitle('True positive rate by demographic group — 5-fold CV', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig7_tpr_by_group.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig7_tpr_by_group.png')

In [ ]:
# Fig 8: Calibration by race group
race_groups_plot = [g for g in demo['race_group'].unique()
                    if (demo['race_group']==g).sum() >= 30]

fig, axes = plt.subplots(3, len(race_groups_plot), figsize=(4*len(race_groups_plot), 10))

for row, (m, color) in enumerate([('XGBoost','steelblue'),('Deep NN','seagreen'),('LSTM','darkorange')]):
    probs = oof_probs[m]
    for col, g in enumerate(race_groups_plot):
        ax = axes[row, col]
        mask = (demo['race_group']==g).values
        y_g, p_g = y[mask], probs[mask]
        if y_g.sum() < 5:
            ax.text(0.5, 0.5, 'Too few\ndeaths', ha='center', va='center', transform=ax.transAxes)
            continue
        frac_pos, mean_pred = calibration_curve(y_g, p_g, n_bins=8)
        ax.plot(mean_pred, frac_pos, 's-', color=color, linewidth=2, label=m)
        ax.plot([0,1],[0,1],'k--',alpha=0.4,label='Perfect')
        ax.set_title(f'{m} — {g}', fontsize=9)
        ax.set_xlabel('Predicted probability'); ax.set_ylabel('Observed fraction')
        ax.set_xlim(0,1); ax.set_ylim(0,1); ax.legend(fontsize=7)

plt.suptitle('Calibration by race group — out-of-fold predictions', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig8_calibration_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig8_calibration_by_race.png')

In [ ]:
# Disparity gap table
print('AUROC disparity gap (max - min mean_auroc within group):')
gaps = summary.groupby(['model','dimension'])['mean_auroc'].agg(
    lambda x: x.max() - x.min()
).unstack('dimension').round(3)
print(gaps.to_string())
print('\nSmaller gap = more equitable across groups.')
gaps.to_csv(OUT_DIR / 'table_disparity_gaps.csv')

---
## 8. Investigation. Why Do Disparities Exist?

In [ ]:
# Training set representation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in zip(axes, GROUP_COLS, titles.values()):
    all_groups = sorted(demo[col].dropna().unique())
    x = np.arange(len(all_groups))
    pct = demo[col].value_counts(normalize=True)*100
    ax.bar(x, [pct.get(g,0) for g in all_groups], color='steelblue', edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(all_groups, rotation=30, ha='right')
    ax.set_ylabel('% of cohort'); ax.set_title(f'{title} distribution')

plt.suptitle('Demographic composition — full cohort (training reflects these proportions)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig9_cohort_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# AUROC vs group size 
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(5*len(MODEL_NAMES), 5))

race_sum = summary[summary['dimension']=='race_group']

for ax, m in zip(axes, MODEL_NAMES):
    sub = race_sum[race_sum['model']==m]
    ax.scatter(sub['mean_n'], sub['mean_auroc'], s=80, color=colors[m], zorder=3)
    ax.errorbar(sub['mean_n'], sub['mean_auroc'], yerr=sub['std_auroc'],
                fmt='none', color='black', capsize=4, linewidth=1)
    for _, row in sub.iterrows():
        ax.annotate(row['group'], (row['mean_n'], row['mean_auroc']),
                    textcoords='offset points', xytext=(6,2), fontsize=9)
    ax.set_xlabel('Mean test set size per fold')
    ax.set_ylabel('Mean AUROC ± std')
    ax.set_title(f'{m} — AUROC vs group size')

plt.suptitle('Does group size predict model performance?', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig10_auroc_vs_size.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Measurement frequency vs AUROC
avg_meas_race = meas.groupby(['race_group','vital'])['n_measurements'].mean().groupby('race_group').sum()

fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(5*len(MODEL_NAMES), 5))

for ax, m in zip(axes, MODEL_NAMES):
    sub = race_sum[race_sum['model']==m].set_index('group')
    shared = avg_meas_race.index.intersection(sub.index)
    x = avg_meas_race.loc[shared]
    y_plot = sub.loc[shared, 'mean_auroc']
    err    = sub.loc[shared, 'std_auroc']
    ax.scatter(x, y_plot, s=80, color=colors[m], zorder=3)
    ax.errorbar(x, y_plot, yerr=err, fmt='none', color='black', capsize=4, linewidth=1)
    for g in shared:
        ax.annotate(g, (x[g], y_plot[g]), textcoords='offset points', xytext=(6,2), fontsize=9)
    ax.set_xlabel('Total avg measurements (first 24h)')
    ax.set_ylabel('Mean AUROC ± std')
    ax.set_title(f'{m} — AUROC vs measurement frequency')

plt.suptitle('Does measurement frequency predict model performance by race?', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig11_auroc_vs_measurements.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Bootstrap Confidence Intervals

Using out-of-fold probabilities to compute 95% bootstrap CIs.

In [ ]:
from sklearn.utils import resample

def bootstrap_auroc_ci(y_true, probs, n_bootstrap=1000):
    scores = []
    for _ in range(n_bootstrap):
        idx = resample(np.arange(len(y_true)))
        if y_true[idx].sum() == 0: continue
        scores.append(roc_auc_score(y_true[idx], probs[idx]))
    return np.mean(scores), np.percentile(scores, 2.5), np.percentile(scores, 97.5)

print('Bootstrap 95% CI for AUROC by race group (out-of-fold probabilities)...')
print('=' * 65)

boot_records = []
for m in MODEL_NAMES:
    probs = oof_probs[m]
    print(f'\n{m}:')
    for group in sorted(demo['race_group'].dropna().unique()):
        mask = (demo['race_group']==group).values
        if mask.sum() < 30 or y[mask].sum() < 5: continue
        mean, lo, hi = bootstrap_auroc_ci(y[mask], probs[mask])
        print(f'  {group:<12}  {mean:.4f}  95% CI [{lo:.4f}, {hi:.4f}]')
        boot_records.append({'model':m,'group':group,'auroc':mean,'ci_lower':lo,'ci_upper':hi})

boot_df = pd.DataFrame(boot_records)
boot_df.to_csv(OUT_DIR / 'bootstrap_ci.csv', index=False)
print('\nSaved bootstrap_ci.csv')

In [ ]:
fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(5*len(MODEL_NAMES), 5))

for ax, m in zip(axes, MODEL_NAMES):
    sub = boot_df[boot_df['model']==m].sort_values('auroc', ascending=False)
    y_pos = np.arange(len(sub))
    ax.barh(y_pos, sub['auroc'], color=colors[m], alpha=0.7, height=0.5)
    ax.errorbar(sub['auroc'], y_pos,
                xerr=[sub['auroc']-sub['ci_lower'], sub['ci_upper']-sub['auroc']],
                fmt='none', color='black', capsize=5, linewidth=2)
    overall_auroc = roc_auc_score(y, oof_probs[m])
    ax.axvline(overall_auroc, color='crimson', linestyle='--', linewidth=1.5,
               label=f'Overall ({overall_auroc:.3f})')
    ax.set_yticks(y_pos); ax.set_yticklabels(sub['group'])
    ax.set_xlabel('AUROC (95% bootstrap CI)')
    ax.set_title(f'{m}'); ax.legend(fontsize=9)

plt.suptitle('AUROC with 95% bootstrap confidence intervals by race group', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig12_bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig12_bootstrap_ci.png')

---
## 10. Summary of Findings

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

# Overall ROC and PR curves using out-of-fold probabilities
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for m, color in [('XGBoost','steelblue'),('Deep NN','seagreen'),('LSTM','darkorange')]:
    probs = oof_probs[m]
    auroc = roc_auc_score(y, probs)
    auprc = average_precision_score(y, probs)
    fpr, tpr, _ = roc_curve(y, probs)
    prec, rec, _ = precision_recall_curve(y, probs)
    axes[0].plot(fpr, tpr, color=color, linewidth=2, label=f'{m} (AUROC={auroc:.3f})')
    axes[1].plot(rec, prec, color=color, linewidth=2, label=f'{m} (AUPRC={auprc:.3f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC curve — out-of-fold'); axes[0].legend()

axes[1].axhline(y.mean(), color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({y.mean():.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-recall curve — out-of-fold'); axes[1].legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'fig13_overall_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig13_overall_performance.png')

In [ ]:
print('=' * 60)
print('THESIS RESULTS SUMMARY — 5-FOLD CV')
print('=' * 60)

print(f'\nCohort: {len(cohort):,} patients  |  Mortality: {cohort["hospital_expire_flag"].mean():.1%}')
print(f'White patients: {(cohort["race_group"]=="White").mean():.1%} of cohort')

print(f'\nOverall performance (mean ± std across 5 folds):')
print(f'{"Model":<12} {"AUROC":>16} {"AUPRC":>16}')
print('-'*48)
for m in MODEL_NAMES:
    row = summary[(summary['model']==m)&(summary['group']=='Overall')].iloc[0]
    print(f'{m:<12}  {row["mean_auroc"]:.4f} ± {row["std_auroc"]:.4f}  '
          f'{row["mean_auprc"]:.4f} ± {row["std_auprc"]:.4f}')

print(f'\nAUROC disparity gap (max-min mean_auroc by race):')
for m in MODEL_NAMES:
    sub = summary[(summary['model']==m)&(summary['dimension']=='race_group')]
    gap = sub['mean_auroc'].max() - sub['mean_auroc'].min()
    print(f'  {m:<12}  {gap:.4f}')

print(f'\nBlack vs White AUROC (out-of-fold):')
for m in MODEL_NAMES:
    sub = boot_df[boot_df['model']==m].set_index('group')
    if 'Black' in sub.index and 'White' in sub.index:
        print(f'  {m:<12}  Black={sub.loc["Black","auroc"]:.4f} '
              f'[{sub.loc["Black","ci_lower"]:.4f},{sub.loc["Black","ci_upper"]:.4f}]  '
              f'White={sub.loc["White","auroc"]:.4f} '
              f'[{sub.loc["White","ci_lower"]:.4f},{sub.loc["White","ci_upper"]:.4f}]')

print(f'\nAll outputs saved to: {OUT_DIR}')
for f in sorted(OUT_DIR.glob('*')):
    print(f'  {f.name}')